# Full tournament simulation

This notebook treats one Monte Carlo draw as one complete World Cup: group stage, group ranking, best third-place selection, bracket construction, and knockout simulation.

In [ ]:
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

from src.world_cup_simulator import (
    parse_annex_c_matchups,
    simulate_2026_tournament
    )

In [ ]:
N_SIMULATIONS = 100000
N_JOBS = -1

MATCHES_FILE = "data/matches.csv"
ROSTER_FILE = "data/roster.csv"
REGULATIONS_PDF = "FWC26_regulations_EN.pdf"

third_place_matchups = parse_annex_c_matchups(REGULATIONS_PDF)
len(third_place_matchups)

In [ ]:
def run_full_tournament_simulation(simulation):
    result = simulate_2026_tournament(
        MATCHES_FILE,
        ROSTER_FILE,
        third_place_matchups,
        ternary=True
        )

    return {
        "simulation": simulation,
        "team_results": result["team_results"],
        "stage_results": result["stage_results"]
        }


tournament_runs = Parallel(n_jobs=N_JOBS)(
    delayed(run_full_tournament_simulation)(simulation)
    for simulation in tqdm(range(1, N_SIMULATIONS + 1))
    )

In [ ]:
team_results = pd.DataFrame(
    [
        {"simulation": run["simulation"], **team_result}
        for run in tournament_runs
        for team_result in run["team_results"]
        ]
    )

stage_results = pd.DataFrame(
    [
        {"simulation": run["simulation"], **run["stage_results"]}
        for run in tournament_runs
        ]
    )

In [ ]:
team_results.head(20)

In [ ]:
stage_sets = {
    "make_round_of_32": {
        "Round_of_32",
        "Round_of_16",
        "Quarterfinals",
        "Semifinals",
        "Fourth_place",
        "Third_place",
        "Second_place",
        "Champion"
        },
    "make_round_of_16": {
        "Round_of_16",
        "Quarterfinals",
        "Semifinals",
        "Fourth_place",
        "Third_place",
        "Second_place",
        "Champion"
        },
    "make_quarterfinals": {
        "Quarterfinals",
        "Semifinals",
        "Fourth_place",
        "Third_place",
        "Second_place",
        "Champion"
        },
    "make_semifinals": {
        "Semifinals",
        "Fourth_place",
        "Third_place",
        "Second_place",
        "Champion"
        },
    "make_final": {"Second_place", "Champion"},
    "win_tournament": {"Champion"}
    }

team_forecast = (
    team_results
    .groupby(["group", "team"])
    .agg(
        expected_group_points=("points", "mean"),
        avg_rating_after_group=("rating_after_group", "mean"),
        win_group=("group_rank", lambda values: (values == 1).mean()),
        finish_second=("group_rank", lambda values: (values == 2).mean()),
        finish_third=("group_rank", lambda values: (values == 3).mean())
        )
    )

for column, result_set in stage_sets.items():
    team_forecast[column] = (
        team_results
        .assign(reached=team_results["result"].isin(result_set))
        .groupby(["group", "team"])["reached"]
        .mean()
        )

team_forecast = (
    team_forecast
    .reset_index()
    .sort_values("win_tournament", ascending=False)
    )

In [ ]:
team_forecast

In [ ]:
group_stage_summary = (
    team_results
    .groupby(["group", "team"])
    .agg(
        avg_points=("points", "mean"),
        points_2nd_pct=("points", lambda values: values.quantile(0.025)),
        points_97th_pct=("points", lambda values: values.quantile(0.975))
        )
    .reset_index()
    )

group_stage_summary[
    ["avg_points", "points_2nd_pct", "points_97th_pct"]
    ] = group_stage_summary[
        ["avg_points", "points_2nd_pct", "points_97th_pct"]
        ].round().astype(int)

In [ ]:
group_stage_summary.sort_values(["group", "avg_points"], ascending=[True, False])

In [ ]:
champion_probability = (
    stage_results["Champion"]
    .value_counts(normalize=True)
    .rename_axis("team")
    .reset_index(name="champion_probability")
    )
champion_probability["monte_carlo_se"] = (
    champion_probability["champion_probability"]
    .mul(1 - champion_probability["champion_probability"])
    .div(N_SIMULATIONS)
    .pow(0.5)
    )

In [ ]:
champion_probability

In [ ]:
finish_probability = (
    team_results
    .groupby(["team", "result"])
    .size()
    .div(N_SIMULATIONS)
    .unstack(fill_value=0)
    )

finish_columns = [
    "Group_stage",
    "Round_of_32",
    "Round_of_16",
    "Quarterfinals",
    "Semifinals",
    "Fourth_place",
    "Third_place",
    "Second_place",
    "Champion"
    ]
finish_probability = finish_probability.reindex(
    columns=finish_columns,
    fill_value=0
    )



In [ ]:
finish_probability.sort_values("Champion", ascending=False)

## Export tables and figures

These cells save the forecast tables and generate publication-ready figures for the article.

In [ ]:
from pathlib import Path

output_data_dir = Path("data")
output_data_dir.mkdir(exist_ok=True)

team_forecast.to_csv(output_data_dir / "full_tournament_team_forecast.csv", index=False)
group_stage_summary.to_csv(output_data_dir / "full_tournament_group_stage_summary.csv", index=False)
champion_probability.to_csv(output_data_dir / "full_tournament_champion_probability.csv", index=False)
finish_probability.reset_index().to_csv(output_data_dir / "full_tournament_finish_probability.csv", index=False)

In [ ]:
from src.forecast_visualizations import (
    plot_champion_odds,
    plot_group_advancement_heatmap,
    plot_group_expected_points,
    plot_stage_ladder
    )

figure_dir = Path("figures")
figure_dir.mkdir(exist_ok=True)

plot_champion_odds(champion_probability, figure_dir / "champion_odds.png")
plot_stage_ladder(team_forecast, figure_dir / "stage_ladder.png")
plot_group_expected_points(group_stage_summary, figure_dir / "group_expected_points.png")
plot_group_advancement_heatmap(team_forecast, figure_dir / "group_advancement_heatmap.png")
